<a href="https://colab.research.google.com/github/iperesadko/Filtering-PC-Parts/blob/main/Filtering-PC-Parts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1

In [1]:
!pip install pyngrok --quiet

2

In [2]:
import os
os.makedirs("pc_parts_filter/app/templates", exist_ok=True)
os.makedirs("pc_parts_filter/app/static", exist_ok=True)
os.makedirs("pc_parts_filter/scripts", exist_ok=True)
os.makedirs("pc_parts_filter/tests", exist_ok=True)
%cd pc_parts_filter

/content/pc_parts_filter


3

In [3]:
%%writefile requirements.txt
fastapi==0.104.1
uvicorn==0.24.0
sqlalchemy==2.0.23
pydantic==2.5.0
python-multipart==0.0.6
aiofiles==23.2.1
jinja2==3.1.2
pytest==7.4.3
pytest-cov==4.1.0

Writing requirements.txt


4

In [4]:
!pip install -r requirements.txt --quiet --upgrade
print("Установка завершена. Конфликты версий можно игнорировать.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.9/92.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.5/407.5 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.1/325.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.9/80.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

5

In [5]:
%%writefile app/database.py
from sqlalchemy import create_engine
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

SQLALCHEMY_DATABASE_URL = "sqlite:///./products.db"

engine = create_engine(
    SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False}
)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)

Base = declarative_base()

Writing app/database.py


6

In [6]:
%%writefile app/models.py
from sqlalchemy import Column, Integer, String, Float, JSON
from app.database import Base

class Product(Base):
    __tablename__ = "products"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, index=True)
    brand = Column(String)
    category = Column(String, index=True)
    price = Column(Float)
    description = Column(String, nullable=True)
    attributes = Column(JSON)

Writing app/models.py


7

In [7]:
%%writefile app/schemas.py
from pydantic import BaseModel
from typing import Optional, Dict, Any, List

class FilterRequest(BaseModel):
    category: Optional[str] = None
    price_min: Optional[float] = None
    price_max: Optional[float] = None
    brands: Optional[List[str]] = None
    search_query: Optional[str] = None
    extra: Optional[Dict[str, Any]] = None
    sort_by: Optional[str] = None
    page: int = 1
    page_size: int = 20

class CompareRequest(BaseModel):
    product_ids: List[int]

Writing app/schemas.py


8

In [8]:
%%writefile app/services.py
from sqlalchemy.orm import Session
from sqlalchemy import cast as cast_expr
from sqlalchemy.sql.expression import cast
from sqlalchemy import Float
from app import models, schemas
from typing import List, Dict, Any

class FilterCompareService:

    @staticmethod
    def filter_products(db: Session, filters: schemas.FilterRequest):
        query = db.query(models.Product)

        if filters.category:
            query = query.filter(models.Product.category == filters.category)

        if filters.price_min is not None:
            query = query.filter(models.Product.price >= filters.price_min)
        if filters.price_max is not None:
            query = query.filter(models.Product.price <= filters.price_max)

        if filters.brands and len(filters.brands) > 0:
            query = query.filter(models.Product.brand.in_(filters.brands))

        if filters.search_query:
            search = f"%{filters.search_query}%"
            query = query.filter(
                models.Product.name.ilike(search) |
                models.Product.description.ilike(search)
            )

        if filters.extra:
            for attr_name, attr_value in filters.extra.items():
                if attr_name.endswith("_min"):
                    real_attr = attr_name[:-4]
                    query = query.filter(
                        cast_expr(models.Product.attributes[real_attr], Float) >= attr_value
                    )
                elif attr_name.endswith("_max"):
                    real_attr = attr_name[:-4]
                    query = query.filter(
                        cast_expr(models.Product.attributes[real_attr], Float) <= attr_value
                    )
                else:
                    query = query.filter(models.Product.attributes[attr_name].astext == str(attr_value))

        if filters.sort_by == "price_asc":
            query = query.order_by(models.Product.price.asc())
        elif filters.sort_by == "price_desc":
            query = query.order_by(models.Product.price.desc())
        elif filters.sort_by == "name_asc":
            query = query.order_by(models.Product.name.asc())

        total = query.count()
        offset = (filters.page - 1) * filters.page_size
        products = query.offset(offset).limit(filters.page_size).all()
        return products, total

    @staticmethod
    def compare_products(db: Session, product_ids: List[int]) -> List[Dict[str, Any]]:
        products = db.query(models.Product).filter(models.Product.id.in_(product_ids)).all()
        if len(products) != len(product_ids):
            raise ValueError("Один или несколько товаров не найдены")
        result = []
        for p in products:
            result.append({
                "id": p.id,
                "name": p.name,
                "brand": p.brand,
                "price": p.price,
                "category": p.category,
                "attributes": p.attributes
            })
        return result

    @staticmethod
    def get_distinct_brands(db: Session, category: str = None) -> List[str]:
        query = db.query(models.Product.brand).distinct()
        if category:
            query = query.filter(models.Product.category == category)
        brands = [row[0] for row in query.all() if row[0] is not None]
        return sorted(brands)

Writing app/services.py


9

In [9]:
%%writefile app/routes.py
from fastapi import APIRouter, Depends, HTTPException, Request
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from sqlalchemy.orm import Session
from app.database import SessionLocal
from app import schemas, services
from app.services import FilterCompareService

router = APIRouter()
templates = Jinja2Templates(directory="app/templates")

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@router.get("/", response_class=HTMLResponse)
def home(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})

@router.get("/compare-page", response_class=HTMLResponse)
def compare_page(request: Request):
    return templates.TemplateResponse("compare.html", {"request": request})

@router.get("/api/health")
def health():
    return {"status": "OK"}

@router.post("/api/filter")
def filter_products(filters: schemas.FilterRequest, db: Session = Depends(get_db)):
    products, total = FilterCompareService.filter_products(db, filters)
    return {"products": products, "total": total, "page": filters.page, "page_size": filters.page_size}

@router.post("/api/compare")
def compare_products(req: schemas.CompareRequest, db: Session = Depends(get_db)):
    if len(req.product_ids) < 2:
        raise HTTPException(status_code=400, detail="Выберите хотя бы два товара")
    if len(req.product_ids) > 4:
        raise HTTPException(status_code=400, detail="Можно сравнить не более 4 товаров")
    try:
        result = FilterCompareService.compare_products(db, req.product_ids)
        return {"comparison": result}
    except ValueError as e:
        raise HTTPException(status_code=404, detail=str(e))

@router.get("/api/brands")
def get_brands(category: str = None, db: Session = Depends(get_db)):
    brands = FilterCompareService.get_distinct_brands(db, category)
    return {"brands": brands}

Writing app/routes.py


10

In [10]:
%%writefile app/main.py
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
from app.routes import router

app = FastAPI(title="Модуль фильтрации и сравнения комплектующих")

app.include_router(router)

app.mount("/static", StaticFiles(directory="app/static"), name="static")

Writing app/main.py


11

In [11]:
%%writefile app/templates/index.html
<!DOCTYPE html>
<html>
<head>
    <title>Каталог комплектующих</title>
    <style>
        body { font-family: Arial; margin: 20px; }
        .filters { border: 1px solid #ccc; padding: 10px; margin-bottom: 20px; }
        .product { border-bottom: 1px solid #eee; padding: 8px; }
        .product button { margin-left: 10px; }
        .compare-bar { position: fixed; bottom: 0; left: 0; right: 0; background: #f8f9fa; padding: 10px; border-top: 1px solid #ccc; }
    </style>
</head>
<body>
    <h1>Комплектующие для ПК</h1>
    <div class="filters">
        <h3>Фильтры</h3>
        <input type="text" id="search" placeholder="Поиск по названию">
        <select id="category">
            <option value="">Все категории</option>
            <option value="CPU">Процессоры</option>
            <option value="GPU">Видеокарты</option>
            <option value="RAM">Оперативная память</option>
        </select>
        <input type="number" id="price_min" placeholder="Цена от">
        <input type="number" id="price_max" placeholder="Цена до">
        <button onclick="applyFilters()">Применить</button>
    </div>
    <div id="products"></div>
    <div class="compare-bar" id="compareBar">
        <strong>Сравнение (<span id="compareCount">0</span>/4):</strong>
        <div id="compareList"></div>
        <button onclick="goCompare()">Сравнить</button>
    </div>
    <script>
        let selectedIds = [];

        function applyFilters() {
            const category = document.getElementById('category').value;
            const price_min = parseFloat(document.getElementById('price_min').value) || null;
            const price_max = parseFloat(document.getElementById('price_max').value) || null;
            const search_query = document.getElementById('search').value;
            fetch('/api/filter', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({category, price_min, price_max, search_query, page:1, page_size:50})
            })
            .then(res => res.json())
            .then(data => {
                const div = document.getElementById('products');
                div.innerHTML = data.products.map(p => `
                    <div class="product">
                        <strong>${p.name}</strong> - ${p.brand} - ${p.price} руб.
                        <button onclick="addToCompare(${p.id})">Добавить к сравнению</button>
                    </div>
                `).join('');
            });
        }

        function addToCompare(id) {
            if (!selectedIds.includes(id) && selectedIds.length < 4) {
                selectedIds.push(id);
                updateCompareBar();
            }
        }

        function removeFromCompare(id) {
            selectedIds = selectedIds.filter(i => i !== id);
            updateCompareBar();
        }

        function updateCompareBar() {
            document.getElementById('compareCount').innerText = selectedIds.length;
            const listDiv = document.getElementById('compareList');
            listDiv.innerHTML = selectedIds.map(id => `<span style="margin-right:10px">ID ${id} <button onclick="removeFromCompare(${id})">x</button></span>`).join('');
        }

        function goCompare() {
            if(selectedIds.length < 2) { alert("Выберите хотя бы два товара"); return; }
            localStorage.setItem('compareIds', JSON.stringify(selectedIds));
            window.location.href = '/compare-page';
        }

        applyFilters();
    </script>
</body>
</html>

Writing app/templates/index.html


12

In [12]:
%%writefile app/templates/compare.html
<!DOCTYPE html>
<html>
<head>
    <title>Сравнение товаров</title>
    <style>
        body { font-family: Arial; margin: 20px; }
        table { border-collapse: collapse; width: 100%; }
        th, td { border: 1px solid #ccc; padding: 8px; text-align: left; vertical-align: top; }
        th { background-color: #f2f2f2; }
        .best { background-color: #d4edda; }
        .worst { background-color: #f8d7da; }
    </style>
</head>
<body>
    <h1>Сравнение товаров</h1>
    <div id="comparisonTable"></div>
    <br><a href="/">← Назад к каталогу</a>
    <script>
        const ids = JSON.parse(localStorage.getItem('compareIds') || '[]');
        if(ids.length < 2) {
            document.getElementById('comparisonTable').innerHTML = '<p>Не выбрано товаров для сравнения.</p>';
        } else {
            fetch('/api/compare', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({product_ids: ids})
            })
            .then(res => res.json())
            .then(data => {
                const products = data.comparison;
                let allKeys = new Set();
                products.forEach(p => {
                    if(p.attributes) Object.keys(p.attributes).forEach(k => allKeys.add(k));
                });
                allKeys = Array.from(allKeys);
                let rows = [['Характеристика', ...products.map(p => p.name)]];
                rows.push(['Цена, руб', ...products.map(p => p.price)]);
                rows.push(['Бренд', ...products.map(p => p.brand)]);
                for(let key of allKeys) {
                    rows.push([key, ...products.map(p => p.attributes && p.attributes[key] !== undefined ? p.attributes[key] : '—')]);
                }
                const numericIndices = [];
                for(let i=2; i<rows.length; i++) {
                    const values = rows[i].slice(1).map(v => parseFloat(v));
                    if(values.every(v => !isNaN(v))) {
                        numericIndices.push(i);
                    }
                }
                let html = '<table><thead><tr>';
                for(let col of rows[0]) html += `<th>${col}</th>`;
                html += '</tr></thead><tbody>';
                for(let r=1; r<rows.length; r++) {
                    html += '<tr>';
                    for(let c=0; c<rows[r].length; c++) {
                        let cell = rows[r][c];
                        let className = '';
                        if(numericIndices.includes(r) && c>0) {
                            const values = rows[r].slice(1).map(v => parseFloat(v));
                            const num = parseFloat(cell);
                            if(!isNaN(num)) {
                                if(num === Math.max(...values)) className = 'best';
                                if(num === Math.min(...values)) className = 'worst';
                            }
                        }
                        html += `<td class="${className}">${cell}</td>`;
                    }
                    html += '</tr>';
                }
                html += '</tbody></table>';
                document.getElementById('comparisonTable').innerHTML = html;
            });
        }
    </script>
</body>
</html>

Writing app/templates/compare.html


13

In [13]:
%%writefile scripts/init_demo_data.py
import sys
sys.path.append('.')
from app.database import engine, SessionLocal
from app import models
from app.models import Product

models.Base.metadata.create_all(bind=engine)

db = SessionLocal()
db.query(Product).delete()

demo_products = [
    Product(name="Intel Core i5-12400F", brand="Intel", category="CPU", price=14990, attributes={"cores": 6, "threads": 12, "frequency": 2.5, "max_frequency": 4.4, "socket": "LGA1700", "tdp": 65}),
    Product(name="AMD Ryzen 5 5600X", brand="AMD", category="CPU", price=16800, attributes={"cores": 6, "threads": 12, "frequency": 3.7, "max_frequency": 4.6, "socket": "AM4", "tdp": 65}),
    Product(name="Intel Core i7-12700K", brand="Intel", category="CPU", price=31200, attributes={"cores": 12, "threads": 20, "frequency": 3.6, "max_frequency": 5.0, "socket": "LGA1700", "tdp": 125}),
    Product(name="NVIDIA RTX 3060", brand="NVIDIA", category="GPU", price=29990, attributes={"vram": 12, "vram_type": "GDDR6", "frequency": 1320, "boost_frequency": 1777}),
    Product(name="AMD Radeon RX 6600", brand="AMD", category="GPU", price=26990, attributes={"vram": 8, "vram_type": "GDDR6", "frequency": 1626, "boost_frequency": 2491}),
    Product(name="Kingston Fury 16GB DDR4", brand="Kingston", category="RAM", price=3990, attributes={"capacity": 16, "type": "DDR4", "frequency": 3200, "timings": "CL16"}),
    Product(name="Corsair Vengeance 32GB DDR5", brand="Corsair", category="RAM", price=11990, attributes={"capacity": 32, "type": "DDR5", "frequency": 5200, "timings": "CL40"}),
]

db.add_all(demo_products)
db.commit()
print(f"Добавлено {len(demo_products)} демо-товаров")
db.close()

Writing scripts/init_demo_data.py


14

In [14]:
!python scripts/init_demo_data.py

Добавлено 7 демо-товаров


15

In [16]:
import threading
import uvicorn
from pyngrok import ngrok
import getpass
import time
import subprocess
import sys

# Убедимся, что pyngrok установлен
!pip install pyngrok --quiet

# Проверяем, есть ли уже аутентификация ngrok
try:
    # Попробуем открыть туннель без токена (если уже был)
    test_tunnel = ngrok.connect(8000, "http")
    ngrok.disconnect(test_tunnel.public_url)
    print("ngrok уже аутентифицирован.")
except:
    print("Для использования ngrok нужен токен аутентификации.")
    print("Зарегистрируйтесь на https://ngrok.com (бесплатно) и получите токен.")
    token = getpass.getpass("Введите ваш ngrok authtoken: ")
    !ngrok authtoken {token}
    print("Токен сохранён.")

# Функция для запуска сервера
def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8001, reload=False)

# Запускаем сервер в отдельном потоке
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Ждём запуска сервера
time.sleep(3)

# Открываем туннель ngrok
try:
    public_url = ngrok.connect(8001, "http")
    print(f"\n✅ Сервер успешно запущен!")
    print(f"🔗 Откройте в браузере: {public_url}")
    print(f"📚 API документация: {public_url}/docs")
except Exception as e:
    print(f"Ошибка при запуске ngrok: {e}")
    print("Попробуйте перезапустить ячейку или вручную выполнить 'ngrok authtoken <токен>'")

ERROR:pyngrok.process.ngrok:t=2026-04-04T13:58:35+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-04T13:58:35+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


Для использования ngrok нужен токен аутентификации.
Зарегистрируйтесь на https://ngrok.com (бесплатно) и получите токен.


KeyboardInterrupt: Interrupted by user